In [1]:
import yfinance as yf
from index_maps import *


def get_nifty_sector_and_peers(ticker):
    info = yf.Ticker(ticker).info
    yf_sector = info.get("sector")

    nifty_sector = yf_to_nifty_map.get(yf_sector)

    if not nifty_sector:
        return None, []

    peers = nifty_sector_peers.get(nifty_sector, [])

    return nifty_sector, peers

In [2]:
sector, peers = get_nifty_sector_and_peers("RELIANCE.NS")

print(sector)
print(peers)

NIFTY ENERGY
['RELIANCE.NS', 'ONGC.NS', 'IOC.NS', 'BPCL.NS', 'GAIL.NS']


In [3]:
def compute_roe(stock):
    try:
        financials = stock.financials
        balance_sheet = stock.balance_sheet
        net_income = financials.loc["Net Income"].iloc[0]
        equity = balance_sheet.loc["Stockholders Equity"].iloc[0]

        if equity == 0:
            return None

        return net_income / equity

    except Exception as e:
        print("ROE error:", e)
        return None

In [4]:
def fetch_metrics(tickers):
    import yfinance as yf
    import pandas as pd

    data = []

    for t in tickers:
        stock = yf.Ticker(t)
        info = stock.info

        roe = compute_roe(stock)

        data.append({
            "ticker": t,
            "roe": roe,
            "pe": info.get("trailingPE"),
            "rev_growth": info.get("revenueGrowth"),
            "debt_equity": info.get("debtToEquity"),
            "margin": info.get("profitMargins")
        })

    return pd.DataFrame(data)

In [5]:
fetch_metrics(peers)

,ticker,roe,pe,rev_growth,debt_equity,margin
0,RELIANCE.NS,0.082600,22.218786,0.104,35.651,0.08122
1,ONGC.NS,0.105479,9.388430,0.008,43.805,0.05798
2,IOC.NS,0.072916,5.627647,0.057,72.631,0.04636
3,BPCL.NS,0.163872,5.439389,0.052,56.386,0.05496
4,GAIL.NS,0.146472,12.105746,-0.045,25.373,0.06029


In [6]:
def add_percentile_scores(df):
    for col in ["roe", "rev_growth", "margin"]:
        df[col + "_pct"] = df[col].rank(pct=True)

    # inverse metrics (lower is better)
    for col in ["pe", "debt_equity"]:
        df[col + "_pct"] = 1 - df[col].rank(pct=True)

    return df

In [ ]:
df_with_scores = add_percentile_scores(fetch_metrics(peers))



In [8]:
df_with_scores

,ticker,roe,pe,rev_growth,debt_equity,margin,roe_pct,rev_growth_pct,margin_pct,pe_pct,debt_equity_pct
0,RELIANCE.NS,0.082600,22.218786,0.104,35.651,0.08122,0.4,1.0,1.0,0.0,0.6
1,ONGC.NS,0.105479,9.388430,0.008,43.805,0.05798,0.6,0.4,0.6,0.4,0.4
2,IOC.NS,0.072916,5.627647,0.057,72.631,0.04636,0.2,0.8,0.2,0.6,0.0
3,BPCL.NS,0.163872,5.439389,0.052,56.386,0.05496,1.0,0.6,0.4,0.8,0.2
4,GAIL.NS,0.146472,12.105746,-0.045,25.373,0.06029,0.8,0.2,0.8,0.2,0.8


In [9]:
def factor_scores(df):
    df["quality"] = (df["roe_pct"] + df["margin_pct"]) / 2
    df["growth"] = df["rev_growth_pct"]
    df["valuation"] = df["pe_pct"]
    df["leverage"] = df["debt_equity_pct"]

    return df

In [10]:
def swot_relative(df, ticker):
    row = df[df["ticker"] == ticker].iloc[0]

    swot = {"Strengths": [], "Weaknesses": [], "Opportunities": [], "Threats": []}

    # Strengths (top 30%)
    if row["quality"] > 0.7:
        swot["Strengths"].append("High quality vs peers")

    if row["growth"] > 0.7:
        swot["Strengths"].append("Strong growth vs peers")

    # Weaknesses (bottom 30%)
    if row["leverage"] < 0.3:
        swot["Weaknesses"].append("High debt vs peers")

    if row["valuation"] < 0.3:
        swot["Weaknesses"].append("Expensive vs peers")

    # Opportunities (mid but improving)
    if 0.4 < row["growth"] < 0.7:
        swot["Opportunities"].append("Moderate growth potential")

    # Threats
    if row["quality"] < 0.4:
        swot["Threats"].append("Weak fundamentals vs peers")

    return swot

In [17]:
df = fetch_metrics(peers)
df = add_percentile_scores(df)
df = factor_scores(df)
swot_relative(df, "RELIANCE.NS")

{'Strengths': ['Strong growth vs peers'],
 'Weaknesses': ['Expensive vs peers'],
 'Opportunities': [],
 'Threats': []}

In [ ]:
def swot_absolute(ticker):
    stock = yf.Ticker(ticker)
    info = stock.info
    financials = stock.financials
    balance_sheet = stock.balance_sheet
    cashflow = stock.cashflow

    swot = {
        "Strengths": [],
        "Weaknesses": [],
        "Opportunities": [],
        "Threats": []
    }
    


    # --- Strengths ---
    if info.get("returnOnEquity", 0) > 0.15:
        swot["Strengths"].append("High ROE")

    if info.get("revenueGrowth", 0) > 0.1:
        swot["Strengths"].append("Strong revenue growth")

    if info.get("profitMargins", 0) > 0.15:
        swot["Strengths"].append("Healthy profit margins")

    # --- Weaknesses ---
    if info.get("debtToEquity", 0) > 150:
        swot["Weaknesses"].append("High debt")

    if info.get("freeCashflow", 1) < 0:
        swot["Weaknesses"].append("Negative cash flow")

    # --- Opportunities ---
    if info.get("earningsGrowth", 0) > 0.1:
        swot["Opportunities"].append("Earnings expansion potential")

    # --- Threats ---
    if info.get("beta", 0) > 1.5:
        swot["Threats"].append("High volatility risk")

    if info.get("trailingPE", 0) > 40:
        swot["Threats"].append("Overvaluation risk")

    return swot

In [ ]:
def merge_swot(swot1, swot2):
    final = {}

    for key in ["Strengths", "Weaknesses", "Opportunities", "Threats"]:
        # combine + remove duplicates
        final[key] = list(set(swot1[key] + swot2[key]))

    return final

In [18]:
def swot_analysis_final(ticker):
    import yfinance as yf

    stock = yf.Ticker(ticker)
    info = stock.info

    # --- Absolute SWOT ---
    swot_abs = swot_absolute(ticker)

    # --- Peer-based SWOT ---
    sector, peers = get_nifty_sector_and_peers(ticker)

    df = fetch_metrics(peers)
    df = add_percentile_scores(df)
    df = factor_scores(df)

    swot_rel = swot_relative(df, ticker)

    # --- Merge ---
    final_swot = merge_swot(swot_abs, swot_rel)

    return final_swot

In [19]:
swot_analysis_final("RELIANCE.NS")

{'Strengths': ['Strong revenue growth', 'Strong growth vs peers'],
 'Weaknesses': ['Expensive vs peers'],
 'Opportunities': [],
 'Threats': []}